# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

df = spark.read.csv("/Workspace/Users/karthikprabhu13072000@gmail.com/.bundle/simple_etl_dab/dev/files/fixtures/customers.csv", 
                    header=True, 
                    inferSchema=True)


silver_df = (df.dropDuplicates()
             .filter(col("age").isNotNull())
             .filter(col("salary").isNotNull()))

display(silver_df)

spark.sql("USE SCHEMA karthikprabhu13072000")

silver_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("silver_customers")


In [ ]:
from pyspark.sql.functions import avg, count

gold_df = silver_df.groupBy("city").agg(
    avg("salary").alias("avg_salary"),
    count("customer_id").alias("customer_count")
).orderBy(col("avg_salary").desc())

display(gold_df)

gold_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("gold_city_summary")

In [ ]:
updates_df = spark.read.csv(
    "/Workspace/Users/karthikprabhu13072000@gmail.com/.bundle/simple_etl_dab/dev/files/fixtures/customers_updates.csv",
    header=True,
    inferSchema=True
)

display(updates_df)


In [ ]:
from pyspark.sql.functions import lit, current_date

silver_df = silver_df.withColumn("is_current",lit(True))\
                    .withColumn("start_date", current_date())\
                    .withColumn("end_date", lit(None).cast("date"))

In [ ]:
from pyspark.sql.functions import current_date

existing_df = spark.table("karthikprabhu13072000.silver_customers")

changed_records = existing_df.alias("old").join(
    updates_df.alias("new"),
    on="customer_id"
).filter(
    (col("old.salary") != col("new.salary")) |
    (col("old.age") != col("new.age"))
)

display(changed_records)

In [ ]:
expired_records = changed_records.select(
    col("old.customer_id").alias("customer_id"),
    col("old.name").alias("name"),
    col("old.city").alias("city"),
    col("old.age").alias("age"),
    col("old.salary").alias("salary")
).withColumn("is_current", lit(False)) \
 .withColumn("end_date", current_date())

display(expired_records)

In [ ]:
new_active_records = updates_df.withColumn("is_current", lit(True)) \
    .withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date"))

display(new_active_records)

In [ ]:
final_scd2_df = expired_records.unionByName(new_active_records)

final_scd2_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_scd2")

In [ ]:
display(spark.table("customer_scd2"))